In [ ]:
# NOTEBOOK 29 - live ingestion: real station data in, 24-hour forecasts out (Delhi, WA-TGCN v3)
# Flow:  OpenAQ (live hourly CPCB-network data)  +  Open-Meteo (recent + forecast wind, no key)
#        -> same cleaning / imputation / 26 features as training -> hourly wind graph A(t) from the live wind
#        -> the 7 saved v3 models (CPU, so the GPU stays free) -> live_export/*.json for the frontend.
# Output files (nothing in frontend/ or app.py is touched):
#   live_export/live_frame.json     one frame in the same shape as a playback frame (+ p10/p90 from the 7 seeds)
#   live_export/live_playback.json  the same frame wrapped in a list, drop-in shape for playback.json
#   live_export/live_meta.json      when it ran, data age, how many stations had fresh data, sources, warnings
#   live_export/live_log.csv        one line per run
# Needs the environment variable OPENAQ_API_KEY (same key as notebook 23). The first run fetches about 300 sensor histories
# (about 5-6 minutes because OpenAQ allows 60 requests a minute); later runs only top up the last 8 hours.
# HOW TO CHECK: this cell only sets up options and prints them.
import os, json, time, copy, io, math, pickle, datetime as dt
import numpy as np, pandas as pd, requests
import torch, torch.nn as nn

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
MODEL_DIR, SCALER_DIR = os.path.join(V2_DIR, 'ecotwin_outputs', 'models'), os.path.join(V2_DIR, 'ecotwin_outputs', 'scalers')
LIVE_DIR, CACHE = os.path.join(V2_DIR, 'live_export'), os.path.join(V2_DIR, 'live_cache')
os.makedirs(LIVE_DIR, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
KEY = os.environ.get('OPENAQ_API_KEY', '')
API = 'https://api.openaq.org/v3'
DEVICE = torch.device('cpu')          # one window x 7 small models: CPU is plenty and leaves the GPU free for training
MIN_FRESH_FRACTION = 0.60              # the 'now' hour is the latest hour in which at least this share of stations reported PM2.5
PARAMS = ['pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'relativehumidity', 'temperature']
TO_FEATURE = {'pm25': 'pm2.5', 'pm10': 'pm10', 'no2': 'no2', 'so2': 'so2', 'co': 'co', 'o3': 'o3', 'relativehumidity': 'rh', 'temperature': 'at'}
INPUT_WINDOW, HORIZON, THRESHOLD_KM, CPCB_THRESHOLD = 48, 24, 12.0, 60.0
print('API key set:', bool(KEY), '| device', DEVICE)

In [ ]:
# 1) the 37 Delhi stations and their OpenAQ monitors (matched by coordinates, cached list from notebook 23)
st = pd.read_excel(os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')); st.columns = st.columns.str.strip().str.lower()
NAMES = st['station'].str.strip().tolist(); LAT = st['latitude'].values.astype(float); LON = st['longitude'].values.astype(float); N = len(NAMES)

loc_cache = os.path.join(V2_DIR, 'crosscity_cache', 'india_locations.json')
assert os.path.exists(loc_cache), 'run notebook 23 first (it saves the OpenAQ location list)'
LOCS = {L['id']: L for L in json.load(open(loc_cache))}
SENSOR = {}                                                         # sensor id -> (parameter, units)
for L in LOCS.values():
    for s in L.get('sensors', []):
        SENSOR[s['id']] = ((s.get('parameter') or {}).get('name'), (s.get('parameter') or {}).get('units'))

def km(a, b, c, d):
    p1, p2 = np.radians(a), np.radians(c); x = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(d - b) / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(x))
monitors = [L for L in LOCS.values() if L.get('isMonitor') and not L.get('isMobile') and L.get('coordinates')
            and any((s.get('parameter') or {}).get('name') == 'pm25' for s in L.get('sensors', []))]
LOC_ID = []
for i in range(N):
    d, best = min(((km(LAT[i], LON[i], L['coordinates']['latitude'], L['coordinates']['longitude']), L) for L in monitors), key=lambda t: t[0])
    LOC_ID.append(best['id'] if d <= 1.0 else None)
print(f'stations matched to an OpenAQ monitor within 1 km: {sum(x is not None for x in LOC_ID)} of {N}')
print('unmatched:', [NAMES[i] for i in range(N) if LOC_ID[i] is None])

# HOW TO CHECK: 35 of 37 matched. Unmatched stations are filled from their neighbours like any missing data.

In [ ]:
# 2) OpenAQ helpers with the rate limit respected (60 requests a minute)
def api_get(path, params=None, tries=4):
    for k in range(tries):
        r = requests.get(API + path, headers={'X-API-Key': KEY}, params=params, timeout=60)
        rem, reset = r.headers.get('X-Ratelimit-Remaining'), r.headers.get('X-Ratelimit-Reset')
        if r.status_code == 429:
            time.sleep(float(reset or 30) + 1); continue
        r.raise_for_status()
        if rem is not None and int(rem) <= 2:
            time.sleep(float(reset or 30) + 1)
        return r.json()
    raise RuntimeError('OpenAQ rate limit kept blocking: ' + path)

def pick_sensors(max_age_h=72):
    # for each station and parameter choose the sensor that reported most recently (some monitors carry duplicate or dead sensors)
    out = {}
    now = pd.Timestamp.now(tz='UTC')
    for i, lid in enumerate(LOC_ID):
        if lid is None: continue
        rows = api_get(f'/locations/{lid}/latest')['results']
        best = {}
        for r in rows:
            p = SENSOR.get(r['sensorsId'], (None, None))[0]
            if p in PARAMS:
                t = pd.Timestamp(r['datetime']['utc'])
                if (now - t) <= pd.Timedelta(hours=max_age_h) and (p not in best or t > best[p][0]):
                    best[p] = (t, r['sensorsId'])
        out[i] = {p: int(v[1]) for p, v in best.items()}
    return out

smap_file = os.path.join(CACHE, 'sensor_map.json')
if os.path.exists(smap_file) and time.time() - os.path.getmtime(smap_file) < 12 * 3600:
    SMAP = {int(k): v for k, v in json.load(open(smap_file)).items()}
else:
    t0 = time.time(); SMAP = pick_sensors(); json.dump(SMAP, open(smap_file, 'w'))
    print(f'sensor map built in {time.time()-t0:.0f}s')
have = {p: sum(p in v for v in SMAP.values()) for p in PARAMS}
print('stations with a fresh sensor per parameter:', have)

# HOW TO CHECK: PM2.5 for about 33-35 stations; the other parameters are usually a few fewer.

In [ ]:
# 3) hourly values per sensor, kept in a cache panel that later runs only top up
def fetch_hours(sensor_id, start_utc, end_utc):
    rows, page = [], 1
    while True:
        j = api_get(f'/sensors/{sensor_id}/hours', {'datetime_from': start_utc.strftime('%Y-%m-%dT%H:%M:%SZ'),
                                                    'datetime_to': end_utc.strftime('%Y-%m-%dT%H:%M:%SZ'), 'limit': 1000, 'page': page})
        res = j.get('results', []); rows += res
        if len(res) < 1000: break
        page += 1
    if not rows: return pd.Series(dtype=float)
    t = pd.to_datetime([r['period']['datetimeFrom']['local'] for r in rows], utc=True).tz_convert('Asia/Kolkata').tz_localize(None).floor('h')
    return pd.Series([r['value'] for r in rows], index=t).groupby(level=0).mean()

panel_file = os.path.join(CACHE, 'panel.pkl')
def update_panel(hours_back):
    panel = pd.read_pickle(panel_file) if os.path.exists(panel_file) else {}
    now = pd.Timestamp.now(tz='UTC'); start = now - pd.Timedelta(hours=hours_back)
    now_local = pd.Timestamp.now(tz='Asia/Kolkata').floor('h').tz_localize(None)          # local on-the-hour grid (data are labelled :00 IST)
    grid = pd.date_range(now_local - pd.Timedelta(hours=hours_back), now_local, freq='h')
    t0 = time.time(); calls = 0
    for p in PARAMS:
        df = panel.get(p, pd.DataFrame(columns=range(N), dtype=float))
        df = df.reindex(df.index.union(grid))
        for i, sm in SMAP.items():
            if p not in sm: continue
            s = fetch_hours(sm[p], start.to_pydatetime(), now.to_pydatetime() + dt.timedelta(hours=1)); calls += 1
            if p == 'co' and SENSOR.get(sm[p], (None, None))[1] == 'µg/m³': s = s / 1000.0       # CPCB CO is mg/m3
            df.loc[s.index.intersection(df.index), i] = s.reindex(s.index.intersection(df.index)).values
        panel[p] = df
    pd.to_pickle(panel, panel_file)
    print(f'panel updated: {calls} sensor calls in {time.time()-t0:.0f}s')
    return panel

panel = update_panel(84 if not os.path.exists(panel_file) else 10)
last = panel['pm25'].dropna(how='all').index.max()
print('latest hour with any PM2.5:', last, '| now (IST):', pd.Timestamp.now(tz='Asia/Kolkata').tz_localize(None).floor('min'))

# HOW TO CHECK: first run about 5-6 minutes. The latest hour is normally within 2-4 hours of now (OpenAQ lags the stations a little).

In [ ]:
# 4) wind: recent hours and the next 3 days from Open-Meteo (one request for all 37 stations, no key)
def fetch_wind():
    p = {'latitude': ','.join(f'{x:.4f}' for x in LAT), 'longitude': ','.join(f'{x:.4f}' for x in LON),
         'hourly': 'wind_speed_10m,wind_direction_10m', 'wind_speed_unit': 'ms', 'timezone': 'Asia/Kolkata', 'past_days': 4, 'forecast_days': 3}
    for k in range(4):
        r = requests.get('https://api.open-meteo.com/v1/forecast', params=p, timeout=90)
        if r.status_code == 200: break
        time.sleep(8 * (k + 1))
    r.raise_for_status()
    js = r.json(); js = js if isinstance(js, list) else [js]
    idx = pd.DatetimeIndex(js[0]['hourly']['time'])
    uv = np.zeros((len(idx), N, 2), dtype=np.float32)
    for i, j in enumerate(js):
        ws = np.array(j['hourly']['wind_speed_10m'], float); wd = np.deg2rad(np.array(j['hourly']['wind_direction_10m'], float) % 360)
        uv[:, i, 0], uv[:, i, 1] = np.nan_to_num(-ws * np.sin(wd)), np.nan_to_num(-ws * np.cos(wd))      # same convention as training
    return idx, uv
W_IDX, W_UV = fetch_wind()
print('wind hours:', W_IDX[0], '->', W_IDX[-1], '| mean speed', float(np.hypot(W_UV[..., 0], W_UV[..., 1]).mean()), 'm/s')

# HOW TO CHECK: the wind covers 4 days back and 3 days ahead. Mean speed about 1-3 m/s in Delhi.

In [ ]:
# 5) cleaning, imputation, features and graph: the same recipe as training (notebooks 01, 03 and 15)
RANGES = {'pm2.5': (0, 1000), 'pm10': (0, 2000), 'no2': (0, 500), 'so2': (0, 500), 'co': (0, 50), 'o3': (0, 500), 'rh': (0, 100), 'at': (-10, 55)}
def geometry():
    dist = np.zeros((N, N)); du = np.zeros((N, N)); dv = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i != j:
                dist[i, j] = km(LAT[i], LON[i], LAT[j], LON[j])
                de = (LON[j] - LON[i]) * np.cos(np.radians(LAT[i])); dn = LAT[j] - LAT[i]; nr = np.hypot(de, dn)
                if nr > 0: du[i, j], dv[i, j] = de / nr, dn / nr
    return dist, du, dv, (dist > 0) & (dist <= THRESHOLD_KM)
DIST, DU, DV, WITHIN = geometry(); STATIC = WITHIN.astype(np.float32)

def impute(P, A):
    P = P.copy(); miss0 = np.isnan(P)
    valid = (~np.isnan(P)).astype(float); v0 = np.nan_to_num(P)
    num, den = v0 @ A.T, valid @ A.T
    P = np.where(np.isnan(P), np.where(den > 0, num / np.maximum(den, 1e-9), np.nan), P)                 # 1 neighbour mean
    with np.errstate(all='ignore'):
        cm = np.nanmean(P, axis=1, keepdims=True)
    P = np.where(np.isnan(P), np.broadcast_to(cm, P.shape), P)                                              # 2 city mean at that hour
    P = pd.DataFrame(P).interpolate(limit_direction='both').ffill().bfill().values                          # 3-4 interpolation, fill
    P = np.where(np.isnan(P), np.nanmedian(P) if np.isfinite(P).any() else 0.0, P)
    return P.astype(np.float32), miss0

def compute_At_vec(u, v, eps=1e-6):
    align = u[:, None] * DU + v[:, None] * DV
    A = np.where(WITHIN & (align > 0.0), align / (DIST + eps), 0.0).astype(np.float32)
    rs = A.sum(1, keepdims=True); rs[rs == 0] = 1.0
    return A / rs

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
TIME_COLS = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend']
FUT_IDX = [FEATURE_COLS.index(c) for c in TIME_COLS + ['u_wind', 'v_wind', 'wspd']]
feat_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
log_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)

def calendar(idx):
    hh, dow, mon = idx.hour.values, idx.dayofweek.values, idx.month.values
    return {'hour_sin': np.sin(2*np.pi*hh/24), 'hour_cos': np.cos(2*np.pi*hh/24), 'dow_sin': np.sin(2*np.pi*dow/7), 'dow_cos': np.cos(2*np.pi*dow/7),
            'mon_sin': np.sin(2*np.pi*(mon-1)/12), 'mon_cos': np.cos(2*np.pi*(mon-1)/12), 'is_weekend': (dow >= 5).astype(float)}
print('recipe ready | features', F)

In [ ]:
class WATGCNv3Ablate(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon,
                 n_stations, dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        self.register_buffer('static_adj', sa, persistent=False)

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(h))) + h
        else:
            if self.graph_mode == 'dynamic':
                A = self._n(At + I.view(1, 1, n, n))
            else:
                A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        else:
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        fut_emb = self.fut_enc(Fut)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, fut_emb], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

In [ ]:
# 6) the 7 saved v3 models (dynamic wind graph), on CPU
SEEDS7 = cfg['seeds']
MODELS = []
for sd in SEEDS7:
    m = WATGCNv3Ablate(F, cfg['hidden_dim'], cfg['gru_layers'], cfg['attn_heads'], HORIZON, N, cfg['dropout'], cfg['n_fut'], cfg['gcn_hops'],
                       graph_mode='dynamic', static_adj=STATIC).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v3_seed{sd}.pt'), map_location=DEVICE)); m.eval(); MODELS.append(m)
print(f'{len(MODELS)} models loaded (seeds {SEEDS7})')

def aqi_category(v):
    return 'Good' if v <= 30 else 'Satisfactory' if v <= 60 else 'Moderate' if v <= 90 else 'Poor' if v <= 120 else 'Very Poor' if v <= 250 else 'Severe'

def forecast_now(panel):
    pm = panel['pm25']
    n_map = sum(x is not None for x in LOC_ID)
    ok = pm.notna().sum(axis=1) >= MIN_FRESH_FRACTION * n_map
    H0 = ok[ok].index.max()                                            # the 'now' hour: latest hour with enough stations reporting
    idx = pd.date_range(H0 - pd.Timedelta(hours=71), H0, freq='h')     # 24 hours of warm-up for the rolling features + the 48-hour input
    raw = {}; obs_mask = None
    for p in PARAMS:
        f = TO_FEATURE[p]; P = panel[p].reindex(idx).reindex(columns=range(N)).values.astype(float)
        lo, hi = RANGES[f]; P = np.where((P >= lo) & (P < hi), P, np.nan)
        P, miss = impute(P, STATIC); raw[f] = P
        if f == 'pm2.5': obs_mask = ~miss
    pmv = pd.DataFrame(raw['pm2.5'])
    der = {'pm_roll6_mean': pmv.rolling(6, min_periods=1).mean().values, 'pm_roll6_std': pmv.rolling(6, min_periods=1).std().fillna(0.0).values,
           'pm_roll24_mean': pmv.rolling(24, min_periods=1).mean().values, 'pm_roll24_std': pmv.rolling(24, min_periods=1).std().fillna(0.0).values,
           'pm_delta3': (pmv - pmv.shift(3)).fillna(0.0).values}
    cm = pmv.mean(axis=1).values[:, None]
    der['city_mean_pm'] = np.broadcast_to(cm, pmv.shape); der['city_max_pm'] = np.broadcast_to(pmv.max(axis=1).values[:, None], pmv.shape)
    der['pm_minus_city'] = pmv.values - cm
    for k, v in calendar(idx).items(): der[k] = np.broadcast_to(v[:, None], pmv.shape)
    wpos = W_IDX.get_indexer(idx); assert (wpos >= 0).all(), 'wind does not cover the input hours'
    uv_h = W_UV[wpos]
    X = np.zeros((len(idx), N, F), dtype=np.float32)
    for fi, col in enumerate(FEATURE_COLS):
        if col in raw: X[:, :, fi] = raw[col]
        elif col == 'u_wind': X[:, :, fi] = uv_h[..., 0]
        elif col == 'v_wind': X[:, :, fi] = uv_h[..., 1]
        elif col == 'wspd': X[:, :, fi] = np.hypot(uv_h[..., 0], uv_h[..., 1])
        else: X[:, :, fi] = der[col]
    fidx = pd.date_range(H0 + pd.Timedelta(hours=1), periods=HORIZON, freq='h')
    fpos = W_IDX.get_indexer(fidx); assert (fpos >= 0).all(), 'wind forecast does not cover the next 24 hours'
    uv_f, uv_in = W_UV[fpos], uv_h[-INPUT_WINDOW:]
    cal_f = calendar(fidx); anchor = to_log(raw['pm2.5'][-1])
    t = lambda a: torch.from_numpy(np.ascontiguousarray(a))[None].to(DEVICE)
    def predict(uvi, uvf):                                                                 # (7 seeds, 24, N) in ug/m3 for a given wind
        X48 = X[-INPUT_WINDOW:].copy()
        X48[:, :, FEATURE_COLS.index('u_wind')], X48[:, :, FEATURE_COLS.index('v_wind')] = uvi[..., 0], uvi[..., 1]
        X48[:, :, FEATURE_COLS.index('wspd')] = np.hypot(uvi[..., 0], uvi[..., 1])
        Xs = feat_scaler.transform(X48.reshape(-1, F)).reshape(INPUT_WINDOW, N, F).astype(np.float32)
        At = np.stack([compute_At_vec(uvi[k, :, 0], uvi[k, :, 1]) for k in range(INPUT_WINDOW)])
        Afc = np.stack([compute_At_vec(uvf[k, :, 0], uvf[k, :, 1]) for k in range(HORIZON)])
        fut_raw = np.zeros((HORIZON, N, F), dtype=np.float32)
        for k, v in cal_f.items(): fut_raw[:, :, FEATURE_COLS.index(k)] = v[:, None]
        fut_raw[:, :, FEATURE_COLS.index('u_wind')], fut_raw[:, :, FEATURE_COLS.index('v_wind')] = uvf[..., 0], uvf[..., 1]
        fut_raw[:, :, FEATURE_COLS.index('wspd')] = np.hypot(uvf[..., 0], uvf[..., 1])
        Fut = feat_scaler.transform(fut_raw.reshape(-1, F)).reshape(HORIZON, N, F)[:, :, FUT_IDX].astype(np.float32)
        with torch.no_grad():
            outs = np.stack([m(t(Xs), t(At), t(Afc), t(Fut), t(anchor)).cpu().numpy()[0] for m in MODELS])      # (7, 24, N) log space
        return from_log(outs)
    P = predict(uv_in, uv_f)                                                                                 # ug/m3
    def perturbed(kind):                                                                   # same scenarios as notebook 12, applied to input AND forecast wind
        u, v, uf, vf = uv_in[..., 0].copy(), uv_in[..., 1].copy(), uv_f[..., 0].copy(), uv_f[..., 1].copy()
        if kind == 'wind_x2': u, v, uf, vf = u * 2.0, v * 2.0, uf * 2.0, vf * 2.0
        elif kind == 'wind_nw':
            s1, s2 = np.hypot(u, v), np.hypot(uf, vf); u, v, uf, vf = s1 * 0.7071, -s1 * 0.7071, s2 * 0.7071, -s2 * 0.7071      # wind FROM the north-west
        elif kind == 'wind_rev': u, v, uf, vf = -u, -v, -uf, -vf
        return np.stack([u, v], -1).astype(np.float32), np.stack([uf, vf], -1).astype(np.float32)
    labels = {'wind_x2': 'Wind doubled', 'wind_nw': 'NW inflow (stubble smoke)', 'wind_rev': 'Wind reversed'}
    whatif = {'scenarios': [{'key': k, 'label': labels[k]} for k in labels], 'cpcb_threshold': CPCB_THRESHOLD, 'mode': 'live', 'windows': {'0': {}}}
    for k in labels:
        pk = predict(*perturbed(k)).mean(0)                                                # (24, N) ensemble mean
        whatif['windows']['0'][k] = {'predicted_24h': np.round(pk).astype(int).T.tolist(), 'peak': np.round(pk.max(0)).astype(int).tolist(),
                                     'breach_h': [int(next((h + 1 for h, v_ in enumerate(pk[:, i]) if v_ > CPCB_THRESHOLD), 0)) for i in range(N)]}
    last_obs_h = {i: (None if not pm[i].notna().any() else float((H0 - pm[i].dropna().index.max()) / pd.Timedelta(hours=1))) for i in range(N)}
    stations = []
    for i in range(N):
        mean = P[:, :, i].mean(0); p10, p90 = np.percentile(P[:, :, i], 10, axis=0), np.percentile(P[:, :, i], 90, axis=0)
        cur = float(raw['pm2.5'][-1, i])
        stations.append({'station_id': i, 'current_pm25': round(cur, 1), 'current_aqi': aqi_category(cur), 'actual_24h': [],
                         'predicted_24h': [round(float(x), 1) for x in mean], 'peak_predicted': round(float(mean.max()), 1),
                         'hours_to_breach_cpcb': next((h + 1 for h, v in enumerate(mean) if v > CPCB_THRESHOLD), None),
                         'p10_24h': [round(float(x), 1) for x in p10], 'p90_24h': [round(float(x), 1) for x in p90],
                         'observed_now': bool(obs_mask[-1, i]), 'hours_since_last_observation': last_obs_h[i]})
    meta = {'generated_at': pd.Timestamp.now(tz='Asia/Kolkata').strftime('%Y-%m-%d %H:%M:%S %Z'), 'now_hour': H0.strftime('%Y-%m-%d %H:%M'),
            'age_hours': round(float((pd.Timestamp.now(tz='Asia/Kolkata').tz_localize(None) - H0) / pd.Timedelta(hours=1)), 1),
            'stations_total': N, 'stations_mapped': n_map, 'stations_observed_now': int(obs_mask[-1].sum()),
            'model': 'WA-TGCN v3, 7-seed ensemble (dynamic wind graph), forecast wind from Open-Meteo',
            'sources': ['OpenAQ v3 hourly (CPCB network)', 'Open-Meteo forecast API (recent + forecast wind)'],
            'notes': ['actual_24h is empty in live frames (the future is not known yet)',
                      'p10_24h / p90_24h are the 10th / 90th percentile across the 7 seeds (ensemble spread, not a calibrated interval)',
                      'training wind was ERA5 reanalysis; live wind is Open-Meteo recent analysis and forecast']}
    if meta['stations_observed_now'] < 0.5 * N: meta['warning'] = 'fewer than half of the stations reported in the latest hour; forecasts lean on imputed values'
    if meta['age_hours'] > 6: meta['warning'] = 'latest data is more than 6 hours old'
    frame = {'window_idx': 0, 'mode': 'live', 'timestamp': H0.strftime('%Y-%m-%d %H:%M'), 'stations': stations}
    return frame, meta, whatif
print('forecast function ready')

In [ ]:
# 7) one live cycle: top up the data, fetch wind, forecast, write the JSON files
PUBLISH_DIR = None     # keep None. Later, when we agree to wire the frontend, set it to the frontend's public data folder and the files are copied there too
def write_outputs(frame, meta, whatif):
    json.dump(frame, open(os.path.join(LIVE_DIR, 'live_frame.json'), 'w'))
    json.dump([frame], open(os.path.join(LIVE_DIR, 'live_playback.json'), 'w'))
    json.dump(meta, open(os.path.join(LIVE_DIR, 'live_meta.json'), 'w'), indent=2)
    json.dump(whatif, open(os.path.join(LIVE_DIR, 'live_whatif.json'), 'w'), separators=(',', ':'))
    log = os.path.join(LIVE_DIR, 'live_log.csv')
    pd.DataFrame([{'generated_at': meta['generated_at'], 'now_hour': meta['now_hour'], 'age_hours': meta['age_hours'],
                   'observed_now': meta['stations_observed_now'], 'mean_now': np.mean([s['current_pm25'] for s in frame['stations']]),
                   'mean_peak': np.mean([s['peak_predicted'] for s in frame['stations']])}]).to_csv(log, mode='a', header=not os.path.exists(log), index=False)
    if PUBLISH_DIR:
        import shutil
        for f in ('live_frame.json', 'live_playback.json', 'live_meta.json', 'live_whatif.json'):
            shutil.copy(os.path.join(LIVE_DIR, f), os.path.join(PUBLISH_DIR, f))

def run_once():
    global panel, W_IDX, W_UV
    panel = update_panel(10)
    W_IDX, W_UV = fetch_wind()
    frame, meta, whatif = forecast_now(panel)
    write_outputs(frame, meta, whatif)
    return frame, meta

frame, meta, whatif = forecast_now(panel)          # first cycle uses the data already loaded above
write_outputs(frame, meta, whatif)
print(json.dumps(meta, indent=1))
cur = np.array([s['current_pm25'] for s in frame['stations']]); pk = np.array([s['peak_predicted'] for s in frame['stations']])
print(f"now ({frame['timestamp']}): mean PM2.5 {cur.mean():.0f} ug/m3 | forecast peak in the next 24 h: mean {pk.mean():.0f}, max {pk.max():.0f}")
print('stations forecast to breach 60 ug/m3:', sum(s['hours_to_breach_cpcb'] is not None for s in frame['stations']), 'of', N)
base_mean = np.array([s['predicted_24h'] for s in frame['stations']]).mean()
for k, e in whatif['windows']['0'].items():
    print(f"  what-if {k:9}: mean forecast {np.mean(e['predicted_24h']):6.1f} ug/m3 ({np.mean(e['predicted_24h']) - base_mean:+.1f} vs baseline {base_mean:.1f}) | stations breaching {sum(1 for h in e['breach_h'] if h > 0)} of {N}")
for s in sorted(frame['stations'], key=lambda s: -s['peak_predicted'])[:5]:
    print(f"  {NAMES[s['station_id']][:28]:28} now {s['current_pm25']:6.1f} | peak {s['peak_predicted']:6.1f} | next 6 h {s['predicted_24h'][:6]}")
print('written to', LIVE_DIR)

# HOW TO CHECK: now_hour is within a few hours of the clock, stations_observed_now is about 30+, current PM2.5 looks like today's
# Delhi values, the what-if lines show how doubled, NW and reversed wind change the forecast, and the 5 stations listed are plausible hotspots.

In [ ]:
# 8) sanity: are the live feature values on the same scale as the training data?
xi = {}
for p in PARAMS:
    f = TO_FEATURE[p]; v = panel[p].iloc[-48:].values.astype(float); xi[f] = float(np.nanmean(v)) if np.isfinite(v).any() else float('nan')
print(f"{'feature':8}{'live mean (48 h)':>18}{'Delhi training mean':>21}")
for f, v in xi.items():
    print(f'{f:8}{v:>18.2f}{feat_scaler.mean_[FEATURE_COLS.index(f)]:>21.2f}')

# HOW TO READ: live values need not match the training average (winter evenings are higher), but a 10x difference in a gas
# (no2, so2, co) would mean a units problem - tell me if you see that.

In [ ]:
# 9) keep it updating: run this cell and leave it open (stop with the Interrupt button). Each cycle refreshes data and rewrites the JSON.
EVERY_MINUTES = 60
def run_loop(every_minutes=EVERY_MINUTES):
    while True:
        try:
            frame, meta = run_once()
            print(time.strftime('%H:%M:%S'), 'updated | now hour', meta['now_hour'], '| observed', meta['stations_observed_now'], '/', N, '|', meta.get('warning', 'ok'))
        except Exception as e:
            print(time.strftime('%H:%M:%S'), 'update failed, will retry next cycle:', repr(e)[:160])
        time.sleep(every_minutes * 60)
# run_loop()      # remove the leading # to start; the notebook (and the laptop) must stay awake and online

In [ ]:
# 10) fallback if the internet or OpenAQ is down: replay a stored Delhi test window as a frame, clearly flagged as a replay
def replay_once(k=None):
    pb = json.load(open(os.path.join(V2_DIR, 'dashboard_export', 'playback.json')))
    k = int(time.time() // 3600) % len(pb) if k is None else k
    fr = dict(pb[k]); fr['mode'] = 'replay'
    json.dump(fr, open(os.path.join(LIVE_DIR, 'live_frame.json'), 'w')); json.dump([fr], open(os.path.join(LIVE_DIR, 'live_playback.json'), 'w'))
    json.dump({'generated_at': pd.Timestamp.now(tz='Asia/Kolkata').strftime('%Y-%m-%d %H:%M:%S %Z'), 'mode': 'replay',
               'notes': ['REPLAY of a stored held-out test window, not live data']}, open(os.path.join(LIVE_DIR, 'live_meta.json'), 'w'), indent=2)
    return fr['timestamp']
# replay_once()   # remove the leading # to use